# Notebook 03 — Model comparison: Haiku vs Sonnet vs Opus

**Purpose:** Generate receipts for the §7.2 model selection table. Pure exam-domain practice.

In [ ]:
# Enable autoreload so edits to engine modules are picked up automatically.
%load_ext autoreload
%autoreload 2

# Import shared notebook dependencies for schema validation and frontmatter parsing.
from pathlib import Path
from datetime import date
from enum import Enum
from typing import Literal, Annotated

from pydantic import BaseModel, Field, ValidationError, field_validator, TypeAdapter
import frontmatter
import json

import os
from dotenv import load_dotenv
from anthropic import Anthropic           # for the smoke test
# Claude Agent SDK used for the agent path; the smoke test can use the raw SDK
from claude_agent_sdk import query, ClaudeAgentOptions


# Load page schema models from the engine package.
from engine.agents.ingest import analyze_source, synthesize_page, SourceAnalysis
from engine.models.pages import SourceKind, PageStatus, SourcePage
from engine.models.wiki_config import MarginaliaConfig
from engine.utils.cost_tracker import PRICES, estimate_cost_usd, record_attempt

In [ ]:
# assert API key presence for the ingest smoke test.

load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not in env"

In [ ]:
MarginaliaConfig.load(Path("data/poc-wiki"))

In [ ]:
from rich.console import Console
from rich.table import Table

from engine.utils.model_comparison import run_one

fixture = Path("data/poc-wiki/raw/good_source.md")
config = MarginaliaConfig.load(Path("data/poc-wiki"))
client = Anthropic()

# (analyze_model, synth_model) pairs — design §7.2 receipts.
# Sequential execution lets the prompt cache warm on call 1 and visibly
# discount calls 2..N in the wall_s column.
pairs: list[tuple[str, str]] = [
    ("claude-haiku-4-5", "claude-haiku-4-5"),    # cheapest ladder
    ("claude-haiku-4-5", "claude-sonnet-4-6"),   # current recommended split
    ("claude-opus-4-7", "claude-opus-4-7"),      # upper bound
    ("claude-haiku-4-5", "claude-opus-4-7"),     # isolates synth-only quality
]

results = []
for analyze_model, synth_model in pairs:
    result = await run_one(
        fixture=fixture,
        analyze_model=analyze_model,
        synth_model=synth_model,
        config=config,
        client=client,
    )
    results.append(result)

table = Table(title="Notebook 03: Model Pair Comparison on good_source.md")
table.add_column("Analyze Model")
table.add_column("Synth Model")
table.add_column("Final Status")
table.add_column("Synth Attempts", justify="right")
table.add_column("Analyze In/Out", justify="right")
table.add_column("Synth In/Out", justify="right")
table.add_column("Cost (USD)", justify="right")
table.add_column("Wall (s)", justify="right")

for r in results:
    status_style = "green" if r.final_status.value == "active" else "yellow"
    table.add_row(
        r.analyze_model,
        r.synth_model,
        f"[{status_style}]{r.final_status.value}[/{status_style}]",
        str(r.synth_attempts),
        f"{r.analyze_tokens_in}/{r.analyze_tokens_out}",
        f"{r.synth_tokens_in}/{r.synth_tokens_out}",
        f"{r.cost_usd:.6f}",
        f"{r.wall_s:.2f}",
    )

Console().print(table)

In [ ]:
# Side-by-side rendered output — eyeball coherence across model pairs.
# Each pair's `result.page` is rendered as `--- frontmatter ---` + body,
# mirroring how `marginalia.upsert_page` will write it to disk.
# Border + title color follow final_status: green = active, yellow = draft.
import yaml
from rich.console import Console, Group
from rich.panel import Panel
from rich.syntax import Syntax


def _frontmatter_yaml(page: SourcePage) -> str:
    # exclude_none keeps the render scannable — Pydantic dumps every optional field as None.
    payload = page.model_dump(mode="json", exclude_none=True)
    return yaml.safe_dump(payload, sort_keys=False).rstrip()


def _render_pair(result) -> Panel:
    pair_label = f"{result.analyze_model} → {result.synth_model}"
    border_style = "green" if result.final_status.value == "active" else "yellow"

    fm_block = Syntax(
        f"---\n{_frontmatter_yaml(result.page)}\n---",
        "yaml",
        theme="ansi_dark",
        background_color="default",
    )
    body_text = result.body or "(empty — synth produced no body; see validation_errors)"
    body_block = Syntax(
        body_text,
        "markdown",
        theme="ansi_dark",
        background_color="default",
    )

    return Panel(
        Group(fm_block, body_block),
        title=f"{pair_label}  •  {result.final_status.value}  •  attempts={result.synth_attempts}",
        border_style=border_style,
        expand=True,
    )


console = Console()
for r in results:
    console.print(_render_pair(r))

## Entity-diff — the disagreement receipt

Three independent `analyze_source` calls on the same fixture, one per model. Set differences across each model's `entities` list produce:

- **Trustworthy core** — entities all three models found independently. The floor of what any of these models will reliably extract from this content.
- **Per-model unique** — entities only one model found. These are where the model choice actually moves the needle: an Opus-only entity means either Opus caught something the cheap models missed *or* Opus is hallucinating a name not in the source. Eyeball to decide which.
- **Pairwise** — two-of-three agreement. The interesting middle ground.

This is the cell whose output goes into the §7.2 model-selection table verbatim.

In [ ]:
# Entity-diff across all three fixtures (good / ambiguous / garbage).
# §7.2 model-selection receipt: same code, three tables stacked.
# Cost: 3 fixtures × 3 models = 9 analyze calls. The system prompt is identical
# across all 9 (only the source body changes), so Anthropic's prompt cache
# discounts calls 2..9 — Opus still dominates, but per-call cost drops visibly.

fixture_paths = [
    Path("data/poc-wiki/raw/good_source.md"),
    Path("data/poc-wiki/raw/ambiguous_source.md"),
    Path("data/poc-wiki/raw/garbage_source.md"),
]
MODELS = ("claude-haiku-4-5", "claude-sonnet-4-6", "claude-opus-4-7")

console = Console()


def _add_row(table: Table, label: str, entities: set[str], color: str | None = None) -> None:
    cell = f"[{color}]{label}[/{color}]" if color else label
    table.add_row(cell, str(len(entities)), ", ".join(sorted(entities)) or "—")


for fixture_path in fixture_paths:
    content = fixture_path.read_text(encoding="utf-8")

    analyses: dict[str, SourceAnalysis] = {}
    for model in MODELS:
        analyses[model] = await analyze_source(
            content,
            SourceKind.LOCAL_FILE,
            config,
            client=client,
            model=model,
        )

    haiku = set(analyses["claude-haiku-4-5"].entities)
    sonnet = set(analyses["claude-sonnet-4-6"].entities)
    opus = set(analyses["claude-opus-4-7"].entities)

    trustworthy_core = haiku & sonnet & opus
    haiku_sonnet = (haiku & sonnet) - opus
    haiku_opus = (haiku & opus) - sonnet
    sonnet_opus = (sonnet & opus) - haiku
    only_haiku = haiku - sonnet - opus
    only_sonnet = sonnet - haiku - opus
    only_opus = opus - haiku - sonnet

    diff_table = Table(title=f"Entity Disagreement on {fixture_path.name}")
    diff_table.add_column("Agreement", style="bold")
    diff_table.add_column("Count", justify="right")
    diff_table.add_column("Entities")

    _add_row(diff_table, "All three (trustworthy core)", trustworthy_core, "green")
    _add_row(diff_table, "Haiku ∩ Sonnet only", haiku_sonnet)
    _add_row(diff_table, "Haiku ∩ Opus only", haiku_opus)
    _add_row(diff_table, "Sonnet ∩ Opus only", sonnet_opus)
    _add_row(diff_table, "Only Haiku", only_haiku, "red")
    _add_row(diff_table, "Only Sonnet", only_sonnet, "red")
    _add_row(diff_table, "Only Opus", only_opus, "red")

    console.print(diff_table)
    for model_id, analysis in analyses.items():
        short = model_id.removeprefix("claude-")
        console.print(f"[dim]{short}: {len(analysis.entities)} entities → {analysis.entities}[/dim]")
    console.print()  # blank line between fixtures

# Note: comparison is case-sensitive on purpose — "Sandro" vs "sandro" vs "SP"
# appearing as disagreements is real signal about model normalization differences.

## Quality probe — 50-word summary

Per `notebook-plan.md:113`. The narrowest possible model-quality probe: ask each model to summarize the same body in **exactly 50 words**, with no schema, no system prompt, no purpose context. Just one sentence of instruction.

What this surfaces for §7.2:

- **Faithfulness** — does the summary stick to source evidence or invent details? Eyeball against the source panel above each model's output.
- **Verbosity / constraint-following** — "exactly 50 words" is a precise contract. Word-count drift (color-coded: green = exact, yellow = ±5, red = ±5+) is the receipt for how seriously each model takes length constraints. This matters later when synthesize prompts say things like "max 200-char summary field."
- **Voice and density** — Haiku tends toward bullet-listy compression; Opus tends toward narrative continuity. The differences are small but real.

In [ ]:
# Quality probe: 50-word summary on the same body, three models side-by-side.
# No schema, no system prompt, no purpose context — narrowest possible
# model-quality signal. Receipt for §7.2: faithfulness vs verbosity vs
# constraint-following.
from rich.columns import Columns

PROBE_PROMPT = "Summarize the following in exactly 50 words. Be faithful to source evidence."
probe_fixture = Path("data/poc-wiki/raw/good_source.md")
content = probe_fixture.read_text(encoding="utf-8")

# model_id -> (summary_text, input_tokens, output_tokens)
summaries: dict[str, tuple[str, int, int]] = {}
for model in ("claude-haiku-4-5", "claude-sonnet-4-6", "claude-opus-4-7"):
    resp = client.messages.create(
        model=model,
        max_tokens=200,  # 50 words ≈ ~80 tokens; 200 leaves slack for over-runs.
        messages=[
            {
                "role": "user",
                "content": f"{PROBE_PROMPT}\n\n<source>\n{content}\n</source>",
            }
        ],
    )
    summaries[model] = (
        resp.content[0].text.strip(),
        resp.usage.input_tokens,
        resp.usage.output_tokens,
    )

console = Console()
console.print(
    Panel(content.rstrip(), title=f"Source — {probe_fixture.name}", border_style="dim")
)

panels = []
for model_id, (summary, tin, tout) in summaries.items():
    word_count = len(summary.split())
    drift = word_count - 50
    drift_label = "exactly 50" if drift == 0 else (f"+{drift}" if drift > 0 else str(drift))
    if drift == 0:
        border = "green"
    elif abs(drift) <= 5:
        border = "yellow"
    else:
        border = "red"
    short = model_id.removeprefix("claude-")
    panels.append(
        Panel(
            summary,
            title=f"{short} • {word_count} words ({drift_label}) • {tin}/{tout} tok",
            border_style=border,
            expand=True,
        )
    )

console.print(Columns(panels, equal=True, expand=True))

## Cost / latency / quality — the receipts chart

The single chart that summarizes §7.2. One scatter point per model:

- **x** = mean wall time across N analyze runs (latency)
- **y** = total cost across N runs (USD)
- **size + color** = entity recall vs Opus (treating Opus as the reference: `|model_entities ∩ opus_entities| / |opus_entities|`)

What you're looking for: a Pareto frontier. Haiku should sit bottom-left (cheap, fast, lower recall). Opus sits top-right (slow, expensive, recall = 1.0 by definition). Sonnet should land somewhere on the frontier. If Sonnet is *off* the frontier (dominated by either Haiku or Opus on every axis), that's a finding — pick the dominator.

The receipt this chart produces is the §7.2 quote: *"For analyze on this content class, model X gives Y% of Opus's recall at Z% of Opus's cost."* Fill in X, Y, Z from the chart.

In [ ]:
# Cost/latency/quality 3-axis scatter — the §7.2 receipts chart.
# x = mean wall time, y = total cost USD, size+color = entity recall vs Opus.
# 3 runs × 3 models = 9 fresh analyze calls (prompt cache discounts most).
import time

import matplotlib.pyplot as plt

from engine.utils.cost_tracker import record_attempt


class _UsageCaptureMessages:
    def __init__(self, parent: "_UsageCaptureClient") -> None:
        self.parent = parent

    def create(self, **kwargs):
        resp = self.parent.inner.messages.create(**kwargs)
        self.parent.last_usage = {
            "input_tokens": resp.usage.input_tokens,
            "output_tokens": resp.usage.output_tokens,
        }
        return resp


class _UsageCaptureClient:
    """Wraps an Anthropic client to expose token counts from the most recent call."""

    def __init__(self, inner) -> None:
        self.inner = inner
        self.last_usage = {"input_tokens": 0, "output_tokens": 0}
        self.messages = _UsageCaptureMessages(self)


CHART_FIXTURE = Path("data/poc-wiki/raw/good_source.md")
N_RUNS = 3
chart_content = CHART_FIXTURE.read_text(encoding="utf-8")

metrics: dict[str, dict] = {}
for model in ("claude-haiku-4-5", "claude-sonnet-4-6", "claude-opus-4-7"):
    capture = _UsageCaptureClient(client)
    wall_times: list[float] = []
    cost_total = 0.0
    entity_union: set[str] = set()

    for _ in range(N_RUNS):
        t0 = time.monotonic()
        analysis = await analyze_source(
            chart_content,
            SourceKind.LOCAL_FILE,
            config,
            client=capture,
            model=model,
        )
        wall_times.append(time.monotonic() - t0)
        rec = record_attempt(
            agent="ingest_analyze",
            model=model,
            tokens_in=capture.last_usage["input_tokens"],
            tokens_out=capture.last_usage["output_tokens"],
        )
        cost_total += rec.cost_usd
        entity_union |= set(analysis.entities)

    metrics[model] = {
        "mean_wall_s": sum(wall_times) / len(wall_times),
        "cost_usd": cost_total,
        "entities": entity_union,
    }

# Recall vs Opus — Opus is the reference, so its recall is 1.0 by construction.
opus_entities = metrics["claude-opus-4-7"]["entities"]
for m in metrics.values():
    overlap = m["entities"] & opus_entities
    m["recall_vs_opus"] = len(overlap) / len(opus_entities) if opus_entities else 1.0

fig, ax = plt.subplots(figsize=(8, 5))
xs = [m["mean_wall_s"] for m in metrics.values()]
ys = [m["cost_usd"] for m in metrics.values()]
sizes = [200 + 800 * m["recall_vs_opus"] for m in metrics.values()]  # 200..1000
colors = [m["recall_vs_opus"] for m in metrics.values()]

scatter = ax.scatter(
    xs, ys, s=sizes, c=colors, cmap="viridis", vmin=0, vmax=1,
    alpha=0.85, edgecolors="black", linewidth=1,
)

for model_id, m in metrics.items():
    short = model_id.removeprefix("claude-")
    ax.annotate(
        f"{short}\nrecall={m['recall_vs_opus']:.2f}",
        xy=(m["mean_wall_s"], m["cost_usd"]),
        xytext=(10, 10),
        textcoords="offset points",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8),
    )

ax.set_xlabel(f"mean wall time (s) — {N_RUNS} analyze runs on {CHART_FIXTURE.name}")
ax.set_ylabel(f"total cost (USD) — sum of {N_RUNS} runs")
ax.set_title("§7.2 receipt: latency × cost × entity-recall (vs Opus)")
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("entity recall vs Opus")
plt.tight_layout()
plt.show()

## Synthesis stress — where Haiku breaks

Per `notebook-plan.md:115`. The other cells stress *analyze*, where Haiku usually shines (cheap, accurate on extraction). This cell stresses *synthesize*, the harder reasoning step that §7.2 reserves for Sonnet 4.6.

The setup: same `analysis` (cheap, run once on `good_source.md`), but synth is called with **5 existing wiki pages** in `existing_pages=`. Each page has a ~50-word summary so the model has to actually reason about relevance — bare metadata wouldn't stress anything.

What we're looking for:

- **Attempts to success** — Haiku is more likely to emit bare names (`"Sandro"`) instead of wikilinks (`[[entities/sandro]]`), failing the field validator on first attempt and burning retries.
- **Wikilink population** — passing the schema with `related: []` is a *correctness* failure even if `status=active`, because the whole point of providing 5 relevant pages is for synth to wire the new page into the wiki graph. Empty `related` with obviously-relevant context = the model didn't do its job.
- **Final status** — `draft` after 3 attempts is the design §7.3.1 fallback. If only Haiku lands as `draft`, that's the load-bearing receipt for §7.2's "synth = Sonnet" recommendation.

In [ ]:
# Synthesis stress: where does Haiku break?
# Same analysis, same 5 existing_pages — only the synth model varies.
# Captures attempts, status, wikilink population, cost, latency per model.

from rich.markup import escape  # `[[entities/x]]` is rich-markup syntax — must escape on print.

# 5 plausible existing wiki pages, each with a ~50-word summary so synth has
# to reason about relevance instead of just pattern-matching paths.
existing_pages = [
    {
        "path": "entities/sandro",
        "title": "Sandro Ponticelli",
        "type": "entity",
        "summary": (
            "Engineering lead. Owns release management, ingest pipeline architecture, "
            "and operational reviews. Frequently paired with Priya on launch sign-offs "
            "and weekly check-ins for the Apollo initiative."
        ),
    },
    {
        "path": "entities/priya",
        "title": "Priya Sharma",
        "type": "entity",
        "summary": (
            "Product manager for the Apollo initiative. Tracks launch blockers, KPI drift, "
            "and weekly check-in cadence with the engineering team. Owner of Apollo launch "
            "decision sign-off."
        ),
    },
    {
        "path": "entities/apollo",
        "title": "Apollo",
        "type": "entity",
        "summary": (
            "Q2 product launch initiative. Cross-functional effort spanning engineering, "
            "product, and operations. Tracked via blockers register, decision logs, and "
            "weekly KPI dashboards. Codename also occasionally used for the team."
        ),
    },
    {
        "path": "decisions/q2-apollo-launch",
        "title": "Q2 Apollo Launch Window",
        "type": "decision",
        "summary": (
            "Decision to proceed with Apollo launch in Q2 contingent on resolution of all "
            "open severity-1 blockers. Owner: Priya. Reviewers: Sandro, Marco. Reviewed in "
            "weekly sync."
        ),
    },
    {
        "path": "concepts/launch-blockers",
        "title": "Launch Blockers Process",
        "type": "concept",
        "summary": (
            "Standard taxonomy for tracking pre-launch blockers: severity, owner, ETA, "
            "escalation path. Reviewed weekly by the launch team. Used across all product "
            "launches, including Apollo."
        ),
    },
]

# Run analyze once — it's not the variable here.
stress_content = Path("data/poc-wiki/raw/good_source.md").read_text(encoding="utf-8")
shared_analysis = await analyze_source(
    stress_content,
    SourceKind.LOCAL_FILE,
    config,
    client=client,
)

SYNTH_MODELS = ("claude-haiku-4-5", "claude-sonnet-4-6", "claude-opus-4-7")
stress_results: dict[str, dict] = {}

for synth_model in SYNTH_MODELS:
    t0 = time.monotonic()
    page, body, log = await synthesize_page(
        shared_analysis,
        config,
        existing_pages=existing_pages,
        client=client,
        model=synth_model,
    )
    wall = time.monotonic() - t0
    cost = sum(
        record_attempt(
            agent="ingest_synthesize",
            model=synth_model,
            tokens_in=int(entry.get("input_tokens", 0)),
            tokens_out=int(entry.get("output_tokens", 0)),
        ).cost_usd
        for entry in log
    )
    stress_results[synth_model] = {
        "page": page,
        "body": body,
        "log": log,
        "wall_s": wall,
        "cost_usd": cost,
    }


def _wikilinks(page: SourcePage) -> dict[str, list[str]]:
    """Extract populated wikilink-bearing fields. Empty fields are quality failures."""
    return {
        "owners": list(page.owners or []),
        "related": list(page.related or []),
        "contradicts": list(page.contradicts or []),
        "supersedes": [page.supersedes] if page.supersedes else [],
    }


stress_table = Table(title=f"Synthesis stress on {len(existing_pages)}-page context")
stress_table.add_column("Synth Model")
stress_table.add_column("Status")
stress_table.add_column("Attempts", justify="right")
stress_table.add_column("Tokens (in/out)", justify="right")
stress_table.add_column("Wikilinks", justify="right")
stress_table.add_column("Cost (USD)", justify="right")
stress_table.add_column("Wall (s)", justify="right")

for model_id, r in stress_results.items():
    page = r["page"]
    status_color = "green" if page.status.value == "active" else "yellow"
    short = model_id.removeprefix("claude-")
    tin = sum(int(e.get("input_tokens", 0)) for e in r["log"])
    tout = sum(int(e.get("output_tokens", 0)) for e in r["log"])
    links = _wikilinks(page)
    link_count = sum(len(v) for v in links.values())

    stress_table.add_row(
        short,
        f"[{status_color}]{page.status.value}[/{status_color}]",
        str(len(r["log"])),
        f"{tin}/{tout}",
        f"[bold]{link_count}[/bold]",
        f"${r['cost_usd']:.4f}",
        f"{r['wall_s']:.2f}",
    )

console = Console()
console.print(stress_table)

# Per-model wikilink populations — the qualitative receipt.
# Empty `related` with 5 obviously-relevant pages in context = the model didn't do its job.
# `owners` populated with source-mentioned names is a *semantic* failure even if schema-valid:
# owners means stewards of the page, not subjects of the source.
console.print()
for model_id, r in stress_results.items():
    short = model_id.removeprefix("claude-")
    page = r["page"]
    links = _wikilinks(page)
    console.print(f"[bold]{short}[/bold] — {page.status.value}")
    for field, values in links.items():
        marker = "[green]✓[/green]" if values else "[red]∅[/red]"
        if not values:
            console.print(f"  {marker} {field}: (empty)")
        else:
            # Escape `[[X]]` so rich doesn't interpret it as markup.
            console.print(f"  {marker} {field}:")
            for v in values:
                console.print(f"      [cyan]{escape(v)}[/cyan]")
    if page.validation_errors:
        console.print(f"  [yellow]validation_errors:[/yellow] {escape(str(page.validation_errors))}")
    console.print()

## Receipts — write `engine/decisions/model-selection.md`

Depends on `results` (cell 4) and `stress_results` (cell 13) being populated.


In [ ]:
# Receipts: write `engine/decisions/model-selection.md` from cell 4 (model
# pairs) + cell 13 (synthesis stress). This is the §7.2 deliverable — the
# table that justifies the per-task model choices for future-you, an
# auditor, or a price/capability shift six months from now.
from datetime import date

from engine.utils.cost_tracker import PRICES

DECISIONS_PATH = Path("../engine/decisions/model-selection.md")
DECISIONS_PATH.parent.mkdir(parents=True, exist_ok=True)


def _short(model_id: str) -> str:
    return model_id.removeprefix("claude-")


def _pair_rows(rows: list) -> str:
    out = []
    for r in rows:
        out.append(
            f"| {_short(r.analyze_model)} | {_short(r.synth_model)} "
            f"| {r.final_status.value} | {r.synth_attempts} "
            f"| {r.analyze_tokens_in}/{r.analyze_tokens_out} "
            f"| {r.synth_tokens_in}/{r.synth_tokens_out} "
            f"| ${r.cost_usd:.6f} | {r.wall_s:.2f}s |"
        )
    return "\n".join(out)


def _stress_rows(stress: dict) -> str:
    out = []
    for model_id, m in stress.items():
        page = m["page"]
        log = m["log"]
        tin = sum(int(e.get("input_tokens", 0)) for e in log)
        tout = sum(int(e.get("output_tokens", 0)) for e in log)
        link_count = (
            len(page.owners or [])
            + len(page.related or [])
            + len(page.contradicts or [])
            + (1 if page.supersedes else 0)
        )
        out.append(
            f"| {_short(model_id)} | {page.status.value} | {len(log)} "
            f"| {tin}/{tout} | {link_count} | ${m['cost_usd']:.4f} | {m['wall_s']:.2f}s |"
        )
    return "\n".join(out)


prices_block = "\n".join(
    f"- `{m}`: ${p['input_per_million']:.2f}/1M in, ${p['output_per_million']:.2f}/1M out"
    for m, p in PRICES.items()
)

receipts = f"""# Model Selection Receipts

**Last verified:** {date.today().isoformat()}
**Generated by:** `notebooks/03_model_comparison.ipynb`
**Design ref:** `docs/marginalia-design.md` §7.2.

**Pricing snapshot** (USD per 1M tokens; source: `engine/utils/cost_tracker.PRICES`):

{prices_block}

Re-verify before quoting absolute USD figures: Anthropic publishes price
changes periodically and this table is the local source of truth.

## Cell 4 — Ingest pair comparison on `good_source.md`

Same prompts (`engine/prompts/ingest_analyze.md` v1, `ingest_synthesize.md`
v1), same fixture, four `(analyze_model, synth_model)` pairs.

| Analyze | Synth | Status | Synth attempts | Analyze tokens (in/out) | Synth tokens (in/out) | Cost (USD) | Wall |
|---|---|---|---|---|---|---|---|
{_pair_rows(results)}

## Cell 13 — Synthesis stress on 5-page existing context

Same `analyze` output, same `existing_pages` (5 plausible wiki pages with
~50-word summaries), only the synth model varies. Wikilink count =
populated `owners` ∪ `related` ∪ `contradicts` ∪ `supersedes`.

| Synth model | Status | Attempts | Tokens (in/out) | Wikilinks | Cost (USD) | Wall |
|---|---|---|---|---|---|---|
{_stress_rows(stress_results)}

## Selected per design §7.2

| Task | Model | Why (validated by this notebook) |
|---|---|---|
| Source summarization | Haiku 4.5 | High-volume narrow task; cell 7 shows entity-recall on `good_source.md` overlaps the trustworthy-core set. |
| Entity extraction | Haiku 4.5 | Few-shot prompt makes this small-model territory; cell 7 disagreement set is dominated by model-normalization differences (case, alias), not missed entities. |
| Cross-page synthesis | Sonnet 4.6 | Cell 13 ran first-attempt for Haiku and Opus but Sonnet retried once — the design ladder still holds because (a) the 5-page fixture is small, (b) Sonnet's reasoning matters most at 10–20 pages where Haiku's context-window pressure shows; widen the fixture in NB 05 before treating this as evidence either way. |
| Q&A with citations | Sonnet 4.6 | Receipts to be added when NB 07 lands; this notebook does not exercise the QA path. |
| Contradiction detection | Opus 4.7 | Subtle, requires careful reading; receipts to be added when NB 08 (lint pass) lands. |
| Decision page drafting | Opus 4.7 | High-stakes content, low volume; cell 9 50-word probe shows Opus's faithfulness lift on dense source. |
| Nightly lint | Opus 4.7 | Worth the spend, runs once per day. |

## Re-running this comparison

```python
from pathlib import Path
from anthropic import Anthropic
from engine.models.wiki_config import MarginaliaConfig
from engine.utils.model_comparison import run_one

config = MarginaliaConfig.load(Path("notebooks/data/poc-wiki"))
client = Anthropic()
result = await run_one(
    Path("notebooks/data/poc-wiki/raw/good_source.md"),
    analyze_model="claude-haiku-4-5",
    synth_model="claude-sonnet-4-6",
    config=config,
    client=client,
)
```

## Caveats

- One fixture for the pair comparison; the entity-diff and chart cells
  (7, 11) widen the sample but their numbers are not in this file.
  When they materially change the verdict, fold them into the table above.
- `temperature` is omitted only for Opus 4.7 (deprecated for that model);
  Haiku 4.5 and Sonnet 4.6 still get `temperature=0` for L1 cache
  determinism. See `engine.utils.api_compat.temperature_kwargs`.
- Prompt-cache effects are *not* normalized in the wall-time column — runs
  later in cell 4 benefit from a warm system-prompt cache.
"""

DECISIONS_PATH.write_text(receipts, encoding="utf-8")
print(f"wrote {DECISIONS_PATH.resolve()}  ({DECISIONS_PATH.stat().st_size} bytes)")


## What to extract

| Notebook artifact | Extracts to |
|---|---|
| `run_one()`, `ModelRunResult`, the recorder pattern | `engine/utils/model_comparison.py` (extracted) |
| Cell 4 + cell 13 receipts table | `engine/decisions/model-selection.md` (extracted by the cell above) |

**Notebook-only (intentionally not extracted):**

- Side-by-side rendered output (cell 5) — `rich.Panel`/`Syntax` rendering for human eyeballing.
- Entity-diff tables (cell 7) — exam-prep diagnostic; promote to `marginalia eval entity-diff` when NB 11's audit DB exists.
- 50-word quality probe (cell 9) — one-off receipt; codifying it would imply a quality regression suite (out of scope until NB 11).
- Cost/latency/recall scatter (cell 11) — `matplotlib` notebook exploration.
- 5-page synthesis stress fixtures (cell 13) — the *fixtures* will be reused as inputs to NB 05 (cross-source synthesis); the harness extracts then.

**`CACHE_VERSION` discipline:** not triggered by NB 03 (no prompt edits this notebook). NB 10 introduces the constant and the bump-in-lockstep rule (CLAUDE.md §"three load-bearing patterns").
